🇬🇧 [English](02_features.ipynb) · 🇪🇸 [Español](02_features_es.ipynb)

# 02 — Features (working notebook, imports `src/fraud`)

Exercises `src/fraud/features.py::build_features` directly, to review the result of
a change to the module without going through the whole training pipeline.


In [1]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents] if (p / "src" / "fraud").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from src.fraud.data import make_split
from src.fraud.features import build_features


In [2]:
split = make_split(PROJECT_ROOT / "data/raw/fraudTrain.csv", PROJECT_ROOT / "data/raw/fraudTest.csv")
train_feat = build_features(split.train)
train_feat[["tx_count_card_1h", "amt_sum_card_24h", "seconds_since_prev_tx",
            "amt_zscore_vs_card_history", "distance_km"]].describe()


,tx_count_card_1h,amt_sum_card_24h,seconds_since_prev_tx,amt_zscore_vs_card_history,distance_km
count,1.037340e+06,1.037340e+06,1.037340e+06,1.037340e+06,1.037340e+06
mean,1.926292e-01,2.790818e+02,3.485081e+04,3.382997e-02,7.609991e+01
std,4.662420e-01,4.299669e+02,9.144182e+04,3.145140e+00,2.911384e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,-1.308383e+03,2.225452e-02
25%,0.000000e+00,6.687000e+01,5.940000e+03,-4.427668e-01,5.532431e+01
50%,0.000000e+00,1.735400e+02,1.643100e+04,-1.925925e-01,7.820873e+01
75%,0.000000e+00,3.498000e+02,4.017200e+04,1.173070e-01,9.847778e+01
max,6.000000e+00,2.977337e+04,2.592000e+06,1.323901e+03,1.521172e+02


Statistics of the derived features on train. `seconds_since_prev_tx` has a floor of
zero (consecutive transactions) and a ceiling at the 30-day sentinel (a card's first
transaction) -- both by construction, not outliers to clean.


In [3]:
# Causal invariant: altering a card's future row must not change an earlier row's
# aggregate for that same card. Same check as tests/test_features.py and the CI
# smoke test, run here on real data.
sample_card = train_feat["cc_num"].value_counts().index[0]
card_rows = split.train[split.train["cc_num"] == sample_card].copy()
before = build_features(card_rows)["tx_count_card_24h"].to_numpy()

card_rows_altered = card_rows.copy()
card_rows_altered.iloc[-1, card_rows_altered.columns.get_loc("amt")] = 999_999.0
after = build_features(card_rows_altered)["tx_count_card_24h"].to_numpy()

assert np.array_equal(before[:-1], after[:-1]), "future row leaked into a past aggregate"
print("OK: altering the last transaction does not change earlier aggregates.")


OK: altering the last transaction does not change earlier aggregates.


Passes: `closed="left"` in `_card_velocity_features` still excludes the
current/future row. This is the first check to run after touching `features.py`.
